In [ ]:
import math

import contextily as ctx
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from matplotlib.patches import FancyArrowPatch
from shapely.geometry import LineString, Point

In [ ]:
df = pd.read_csv("E:\Python\Bicycle_Traffic_Flow_GNN\data\smartcameras_line--2025-01-01T03-01-00--2025-02-01T03-01-00--tud_project--y16mv1.csv")

for col in ["created_at", "start_time", "end_time"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

for col in ["count_in", "count_out"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# OPTIONAL: set your local timezone if needed (data looks naive)
df["start_time"] = df["start_time"].dt.tz_localize("UTC").dt.tz_convert("Europe/Amsterdam")

display(df.head())

In [ ]:
SENSOR_ID = "TUD_SBX01"         # change this
DAY = "2025-01-12"              # change this (YYYY-MM-DD)

m = (
    (df["sensor_id"] == SENSOR_ID)
    & (df["start_time"].dt.date == pd.to_datetime(DAY).date())
)
d = df.loc[m].copy()

if d.empty:
    raise ValueError(f"No rows for sensor {SENSOR_ID!r} on {DAY}.")

In [ ]:
# --- Aggregate ---
d = d.set_index('start_time')
per_min = d[['count_in', 'count_out']].groupby('start_time').sum()
per_5min = per_min.resample('5T').sum()
per_15min = per_min.resample('15T').sum()

In [ ]:
# --- Two plots: per-minute vs per-5-min ---
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)

# Per-minute
axes[0].plot(per_min.index, per_min['count_in'], label="Uitgaand")
axes[0].plot(per_min.index, per_min['count_out'], label="Inkomend")
axes[0].set_title(f"{SENSOR_ID} — {DAY}: Per Minute")
axes[0].set_ylabel("Counts")
axes[0].legend()

# Per-5-minute
axes[1].plot(per_5min.index, per_5min['count_in'], label="Uitgaand (5min)")
axes[1].plot(per_5min.index, per_5min['count_out'], label="Inkomend (5min)")
axes[1].set_title(f"{SENSOR_ID} — {DAY}: Per 5 Minutes")
axes[1].set_ylabel("Counts")
axes[1].legend()

# Per-15-minute
axes[2].plot(per_15min.index, per_15min['count_in'], label="Uitgaand (15min)")
axes[2].plot(per_15min.index, per_15min['count_out'], label="Inkomend (15min)")
axes[2].set_title(f"{SENSOR_ID} — {DAY}: Per 15 Minutes")
axes[2].set_ylabel("Counts")
axes[2].set_xlabel("Time")
axes[2].legend()


# --- Format x-axis without mdates ---
# Use pandas' built-in formatter via index.strftime
axes[2].set_xticks(per_15min.index[::4])   # show every ~hour (12*5min = 60min)
axes[2].set_xticklabels(per_15min.index[::4].strftime("%H:%M"), rotation=45, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
SENSOR_ID_2 = "TUD_SBX01"         # change this
DAY_2 = "2025-01-13"              # change this (YYYY-MM-DD)

m2 = (
    (df["sensor_id"] == SENSOR_ID_2)
    & (df["start_time"].dt.date == pd.to_datetime(DAY_2).date())
)
d2 = df.loc[m2].copy()

if d2.empty:
    raise ValueError(f"No rows for sensor {SENSOR_ID_2!r} on {DAY_2}.")

# --- Aggregate ---
d2 = d2.set_index('start_time')
per_min2 = d2[['count_in', 'count_out']].groupby('start_time').sum()
per_5min2 = per_min2.resample('5T').sum()
per_15min2 = per_min2.resample('15T').sum()

In [ ]:
# --- Two plots: per-minute vs per-5-min ---
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharey=True)

# Per-5-minute
axes[0].plot(per_5min.index, per_5min['count_in'], label="Uitgaand (zondag 12 jan 2025)")
axes[0].plot(per_5min.index, per_5min['count_out'], label="Inkomend (zondag 12 jan 2025)")
axes[0].set_title(f"Sunday — {DAY}: Per 5 Minutes")
axes[0].set_ylabel("Counts")
# axes[0].set_xlabel("Time")
axes[0].tick_params(labelbottom=False)  # hides only the labels, keeps the ticks
axes[0].legend()

# Per-15-minute
axes[1].plot(per_5min2.index, per_5min2['count_in'], label="Uitgaand (maandag 13 jan 2025)")
axes[1].plot(per_5min2.index, per_5min2['count_out'], label="Inkomend (maandag 13 jan 2025)")
axes[1].set_title(f"Monday — {DAY_2}: Per 5 Minutes")
axes[1].set_ylabel("Counts")
axes[1].set_xlabel("Time")
axes[1].legend()


# --- Format x-axis without mdates ---
# Use pandas' built-in formatter via index.strftime
# axes[0].set_xticks(per_5min.index[::12])   # show every ~hour (12*5min = 60min)
# axes[0].set_xticklabels(per_5min.index[::12].strftime("%H:%M"), rotation=45, ha="right")
axes[1].set_xticks(per_5min2.index[::12])   # show every ~hour (12*5min = 60min)
axes[1].set_xticklabels(per_5min2.index[::12].strftime("%H:%M"), rotation=45, ha="right")

plt.tight_layout()
plt.show()

In [ ]:

# ----------------------------- YOUR NETWORK ---------------------------------
NODES = ['A','B','C','D','E','F','G','H','I','J','K','L','M','N','O','P']

# (1) FILL THESE with real lat/lon for each intersection (WGS84; degrees)
# Tip: read coordinates from OSM/Google by right-click → "What's here?"
LATLON = {
    'A': (52.002387461454894, 4.368163144778211),
    'B': (52.00322958830617, 4.371172583293815),
    'C': (52.001816127154484, 4.368554747294482),
    'D': (52.002285084165315, 4.37196651716244),
    'E': (52.00040592398227, 4.369504249285934),
    'F': (52.001162221124005, 4.372620976162166),
    'G': (52.00137028317236, 4.373516833972758),
    'H': (51.998215434938324, 4.369278783700182),
    'I': (51.998620920399176, 4.370767387264709),
    'J': (51.99972071179148, 4.373618288636594),
    'K': (51.99998732382329, 4.374502428935213),
    'L': (52.00037200633484, 4.375946586449633),
    'M': (51.99610066683201, 4.372495594651472),
    'N': (51.99691659630659, 4.375576651602874),
    'O': (51.99809595711493, 4.375706048763282),
    'P': (51.994083636993054, 4.378308115141057),
}

PHYSICAL_EDGES = [
    ('A','B'), ('A','C'), ('C','E'), ('B','D'), ('D','F'), ('C','D'), ('D','G'),
    ('G','K'), ('K','O'), ('H','I'), ('I','J'), ('J','K'), ('K','L'),
    ('E','I'), ('F','J'), ('J','N'), ('N','P'), ('I','M'), ('M','N'), ('N','O')
]

NO_SENSOR_UNDIR = {('A','B'), ('M','N'), ('N','O'), ('D','G'),
                   ('D','F'), ('J','N'), ('C','D')}

SENSOR_TO_EDGE = {
    "TUD_SBX01|Line 0": ("D", "B"),
    "TUD_SPX18|Line 0": ("A", "C"),
    "TUD_SBX22|Line 0": ("E", "C"),
    "TUD_SBXN6A|Line 2": ("F", "J"),
    "TUD_SBXN6A|Line 0": ("J", "K"),
    "TUD_SPX23|Line 0": ("I", "E"),
    "TUD_SBX25|Line 2": ("I", "H"),
    "TUD_SBX26|Line 0": ("I", "M"),
    "TUD_SPX24|Line 0": ("J", "I"),
    "TUD_SBX07-MULTI|Line 1": ("G", "K"),
    "TUD_SBX07-MULTI|Line 2": ("K", "L"),
    "TUD_SBX07-MULTI|Line 3": ("O", "K"),
    "TUD_SBXN13|Line 0": ("P", "N"),
}

# --------------------------- Build GeoDataFrames -----------------------------
def undirected(u, v):  # for set comparisons
    return tuple(sorted((u, v)))

phys_undir = {undirected(u,v) for (u,v) in PHYSICAL_EDGES}
nosensor_undir = {undirected(u,v) for (u,v) in NO_SENSOR_UNDIR}
sensor_undir = phys_undir - nosensor_undir
sensor_directed = set(SENSOR_TO_EDGE.values())

# Nodes GDF (WGS84)
nodes_gdf = gpd.GeoDataFrame(
    {"node": list(LATLON.keys())},
    geometry=[Point(LATLON[n][1], LATLON[n][0]) for n in LATLON],  # Point(lon, lat)
    crs="EPSG:4326"
)

# Edge helpers
def line_geom(u, v):
    lat_u, lon_u = LATLON[u]
    lat_v, lon_v = LATLON[v]
    return LineString([(lon_u, lat_u), (lon_v, lat_v)])

edges_no_gdf = gpd.GeoDataFrame(
    {"u": [u for u,v in nosensor_undir], "v": [v for u,v in nosensor_undir]},
    geometry=[line_geom(u, v) for (u, v) in nosensor_undir],
    crs="EPSG:4326"
)

edges_yes_gdf = gpd.GeoDataFrame(
    {"u": [u for u,v in sensor_undir], "v": [v for u,v in sensor_undir]},
    geometry=[line_geom(u, v) for (u, v) in sensor_undir],
    crs="EPSG:4326"
)

# Project to Web Mercator for basemap tiles
nodes_3857 = nodes_gdf.to_crs(epsg=3857)
edges_no_3857 = edges_no_gdf.to_crs(epsg=3857)
edges_yes_3857 = edges_yes_gdf.to_crs(epsg=3857)

# ------------------------------ Plotting -------------------------------------
fig, ax = plt.subplots(figsize=(7.5, 6.5))

# Edges: no-sensor (dashed gray), sensor (solid red)
edges_no_3857.plot(ax=ax, linewidth=2.0, color="#D62828", linestyle="--",
                   zorder=2, label="Link without sensor")
edges_yes_3857.plot(ax=ax, linewidth=4.0, color="#D62828", linestyle="-",
                    zorder=3, label="Link with sensor")

# Nodes: bigger markers, white fill with black edge
nodes_3857.plot(ax=ax, color="white", edgecolor="black",
                markersize=150,  # <-- bigger nodes (try 100–180)
                linewidth=1.4, zorder=4)

# Labels: centered bold
for _, row in nodes_3857.iterrows():
    ax.text(row.geometry.x, row.geometry.y, row["node"],
            fontsize=10, fontweight="bold", ha="center", va="center", zorder=5)

# Extent padding
xmin, ymin, xmax, ymax = nodes_3857.total_bounds
pad = 20  # meters
ax.set_xlim(xmin - pad, xmax + pad)
ax.set_ylim(ymin - pad, ymax + pad)

# Basemap (lighter style also works: ctx.providers.CartoDB.Positron)
ctx.add_basemap(ax, crs="EPSG:3857", source=ctx.providers.OpenStreetMap.Mapnik)

# Legend (no arrow entry)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, loc="upper right", frameon=True, fontsize=9)

ax.set_axis_off()
plt.tight_layout()
plt.show()